In [6]:
import json
import csv
import torch
from pathlib import Path

In [7]:
"""
Compute mean and std of evaluation metrics across results/api_1 and results/api_2.
Outputs a CSV with columns: model, method, accuracy, f1_macro.
Values are formatted as "mean ± std" in percentage scale.
"""

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR   = Path.cwd().parent  # notebook lives in notebooks/, project root is one level up
API_1_DIR  = BASE_DIR / "results" / "api_1"
API_2_DIR  = BASE_DIR / "results" / "api_2"
OUTPUT_CSV = BASE_DIR / "results" / "api_mean_std.csv"

METRICS = ["accuracy", "f1_macro"]

# ── Collect files present in both directories ──────────────────────────────────
api1_files = {f.name for f in API_1_DIR.glob("*.json")}
api2_files = {f.name for f in API_2_DIR.glob("*.json")}
common_files = sorted(api1_files & api2_files)
only_api1    = sorted(api1_files - api2_files)
only_api2    = sorted(api2_files - api1_files)

if only_api1:
    print(f"[WARNING] Files only in api_1 (skipped): {only_api1}")
if only_api2:
    print(f"[WARNING] Files only in api_2 (skipped): {only_api2}")

print(f"Processing {len(common_files)} matched file(s)...\n")

# ── Compute stats and write CSV ────────────────────────────────────────────────
def mean_std_str(v1, v2, decimals=2):
    """Return 'mean ± std' string in percentage scale using torch."""
    values = torch.tensor([v1, v2], dtype=torch.float64) * 100
    mean   = values.mean().item()
    std    = values.std(correction=1).item()  # sample std (ddof=1)
    return f"{mean:.{decimals}f} ± {std:.{decimals}f}"

rows = []

for fname in common_files:
    stem = fname.replace(".json", "")          # e.g. "gpt_zero-shot_no_rag"
    parts = stem.split("_", 1)                 # split on first "_" only
    model_name = parts[0]                      # e.g. "gpt"
    method     = parts[1] if len(parts) > 1 else stem  # e.g. "zero-shot_no_rag"

    with open(API_1_DIR / fname) as f1, open(API_2_DIR / fname) as f2:
        data1 = json.load(f1)
        data2 = json.load(f2)

    row = {"model": model_name, "method": method}
    skip = False
    for metric in METRICS:
        v1 = data1.get(metric)
        v2 = data2.get(metric)
        if v1 is None or v2 is None:
            print(f"[WARNING] '{metric}' missing in {fname} — skipping row.")
            skip = True
            break
        row[metric] = mean_std_str(v1, v2)

    if not skip:
        rows.append(row)
        print(f"  {model_name:<12}  {method:<25}  acc={row['accuracy']}  f1={row['f1_macro']}")

# ── Write output ───────────────────────────────────────────────────────────────
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_CSV, "w", newline="") as csvfile:
    fieldnames = ["model", "method", "accuracy", "f1_macro"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"\nDone! Results saved to: {OUTPUT_CSV}")


Processing 14 matched file(s)...

  claude        few-shot_no_rag            acc=81.83 ± 0.03  f1=81.72 ± 0.21
  claude        zero-shot_no_rag           acc=78.91 ± 0.22  f1=79.30 ± 0.12
  deepseek      few-shot_no_rag            acc=62.94 ± 0.45  f1=63.93 ± 1.87
  deepseek      zero-shot_no_rag           acc=59.84 ± 0.57  f1=61.36 ± 0.01
  gemini        few-shot_no_rag            acc=82.73 ± 0.03  f1=82.52 ± 0.08
  gemini        zero-shot_no_rag           acc=81.18 ± 0.32  f1=81.50 ± 0.55
  gemini        zero-shot_rag              acc=83.58 ± 2.32  f1=84.41 ± 2.33
  gpt           few-shot_no_rag            acc=76.59 ± 1.34  f1=77.12 ± 0.47
  gpt           zero-shot_no_rag           acc=74.82 ± 0.35  f1=76.30 ± 0.47
  gpt           zero-shot_rag              acc=72.15 ± 0.70  f1=74.46 ± 0.33
  kimi          few-shot_no_rag            acc=78.35 ± 0.83  f1=78.62 ± 2.36
  kimi          zero-shot_no_rag           acc=76.31 ± 0.16  f1=76.90 ± 1.36
  llama         few-shot_no_rag           

In [8]:
# ── Fine-tuned model stats: results/ft_1 vs results/ft_2 ─────────────────
"""
Compute mean and std of evaluation metrics across results/ft_1 and results/ft_2.
Outputs a CSV with columns: model, accuracy, f1_macro, auroc, aupr.
Values are formatted as "mean ± std" in percentage scale.
Note: raw values > 1 are assumed to be in percentage scale and are divided by 100 first.
"""

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR    = Path.cwd().parent  # notebook lives in notebooks/, project root is one level up
FT_1_DIR    = BASE_DIR / "results" / "ft_1"
FT_2_DIR    = BASE_DIR / "results" / "ft_2"
OUTPUT_CSV  = BASE_DIR / "results" / "ft_mean_std.csv"

FT_METRICS  = ["accuracy", "f1_macro", "auroc", "aupr"]

def to_pct(v):
    """Ensure value is in [0, 100] scale: divide by 100 if > 1, then multiply by 100."""
    return (v / 100.0 if v > 1.0 else v) * 100.0

def mean_std_str_ft(v1, v2, decimals=2):
    """Return 'mean ± std' string in percentage scale using torch."""
    values = torch.tensor([to_pct(v1), to_pct(v2)], dtype=torch.float64)
    mean   = values.mean().item()
    std    = values.std(correction=1).item()  # sample std (ddof=1)
    return f"{mean:.{decimals}f} ± {std:.{decimals}f}"

# ── Collect files present in both directories ──────────────────────────────────
ft1_files    = {f.name for f in FT_1_DIR.glob("*.json")}
ft2_files    = {f.name for f in FT_2_DIR.glob("*.json")}
common_files = sorted(ft1_files & ft2_files)
only_ft1     = sorted(ft1_files - ft2_files)
only_ft2     = sorted(ft2_files - ft1_files)

if only_ft1:
    print(f"[WARNING] Files only in ft_1 (skipped): {only_ft1}")
if only_ft2:
    print(f"[WARNING] Files only in ft_2 (skipped): {only_ft2}")

print(f"Processing {len(common_files)} matched file(s)...\n")

# ── Compute stats and write CSV ────────────────────────────────────────────────
ft_rows = []

for fname in common_files:
    model_name = fname.replace(".json", "")

    with open(FT_1_DIR / fname) as f1, open(FT_2_DIR / fname) as f2:
        data1 = json.load(f1)
        data2 = json.load(f2)

    row  = {"model": model_name}
    skip = False
    for metric in FT_METRICS:
        v1 = data1.get(metric)
        v2 = data2.get(metric)
        if v1 is None or v2 is None:
            print(f"[WARNING] '{metric}' missing in {fname} — skipping row.")
            skip = True
            break
        row[metric] = mean_std_str_ft(v1, v2)

    if not skip:
        ft_rows.append(row)
        print(f"  {model_name:<15}  acc={row['accuracy']}  f1={row['f1_macro']}  auroc={row['auroc']}  aupr={row['aupr']}")

# ── Write output ───────────────────────────────────────────────────────────────
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_CSV, "w", newline="") as csvfile:
    fieldnames = ["model", "accuracy", "f1_macro", "auroc", "aupr"]
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(ft_rows)

print(f"\nDone! Results saved to: {OUTPUT_CSV}")


Processing 4 matched file(s)...

  alertaug         acc=81.60 ± 5.11  f1=74.24 ± 7.12  auroc=98.17 ± 0.66  aupr=80.99 ± 6.67
  smet             acc=65.45 ± 0.00  f1=28.64 ± 0.00  auroc=89.43 ± 0.00  aupr=83.16 ± 0.00
  tram             acc=88.54 ± 0.95  f1=83.52 ± 1.91  auroc=99.18 ± 0.11  aupr=90.16 ± 0.98
  ttpc             acc=39.76 ± 0.00  f1=16.87 ± 0.00  auroc=42.72 ± 0.00  aupr=33.67 ± 0.00

Done! Results saved to: /home/pks230000/research/ALERT/results/ft_mean_std.csv
